--------------------------------------------------------------------------------------------------------------------

**This code uses the following datasets:**

    - NOHRSC Snow Analysis (sfav2): 
      http://www.nohrsc.noaa.gov/snowfall_v2/data/
      
    - Inspiration was taken from tomerburg on GitHub. Their work can be found here: 
      https://github.com/tomerburg/python_gallery/blob/master/nohrsc_snow_plotting.py
--------------------------------------------------------------------------------------------------------------------

## NOHRSC 24-Hour Snowfall Accumulation
This script downloads and visualizes the NOHRSC sfav2 24-hour snowfall accumulation data over a user-set time range. This code can perform 4 different plot types:
- `sum`: snowfall over the full date range
- `max`: maximum single-day accumulation at each grid point
- `std`: standard deviation of daily snowfall
- `days`: count of days exceeding a user-defined accumulation threshold

In [ ]:
import os
import urllib.request
import numpy as np
import xarray as xr
import datetime as dt
import matplotlib.pyplot as plt
import matplotlib.colors as col
import cartopy
from cartopy import crs as ccrs
from cartopy import feature as cfeat
from pathlib import Path

## 1.) Date and Plot Specification

All run parameters are set here. No edits should be needed below this cell.

**Date format:** `YYYYMMDD` — all files are valid at **1200 UTC** (NOHRSC standard issue time).

**Plot types:**
- `sum`: snowfall over the full date range
- `max`:  maximum single-day accumulation at each grid point
- `std`: standard deviation of daily snowfall (high = inconsistent, low = consistent snowfall)
- `days`: count of days exceeding a user-defined accumulation threshold `snow_thres`

**Map extents:**
- `conus`: full contiguous United States
- `midwest`: for comparison with the SHELDUS data

In [ ]:
# date range
start_date = "20110201"
end_date = "20110203"

# specify directory to store files in
work_dir = Path("/data1/alamia/SnowAccum/")

# type of plot
plot_type = 'sum'    # 'sum' | 'max' | 'std' | 'days'
snow_thres = 1.0     # inches, only used when plot_type == 'days'
map_extent = 'conus' # 'conus' | 'midwest'

## 2.) Download Snowfall Accumulation Files

NOHRSC sfav2 files represent snowfall accumulated during the **24 hours preceding** the valid time, so the loop starts one day *after* `START_DATE`.

In [ ]:
# convert start & end dates to datetime objects
# 12 UTC, as that is the standard NOHRSC file time
sdate = dt.datetime.strptime(start_date+"12","%Y%m%d%H") # start date
edate = dt.datetime.strptime(end_date+"12","%Y%m%d%H")   # end date

# start iterator one day ahead since each file covers the 24 hours BEFORE its valid time, so the first file needed is the one valid at START_DATE+1 1200 UTC
idate = sdate + dt.timedelta(hours=24)

# empty list to store dates in, which will be used to open these files later
files = []

# iterate through the range of dates provided
while idate <= edate:
    
    # format current date as string
    strdate = dt.datetime.strftime(idate,'%Y%m%d%H')
    yyyymm = dt.datetime.strftime(idate,'%Y%m')
    
    # construct NOHRSC data URL
    url = f"http://www.nohrsc.noaa.gov/snowfall_v2/data/{yyyymm}/sfav2_CONUS_24h_{strdate}.nc"
    savepath = f"{work_dir}{strdate}.nc"

    # download file, if it doesn't exist locally
    if os.path.isfile(savepath) == False:
        urllib.request.urlretrieve(url, savepath)
        print(f"---> Downloaded file for {strdate}")
        
    # append filepath to list of files to open
    files.append(savepath)
    
    # move to next day
    idate += dt.timedelta(hours=24)

## 3.) Load Data, Convert Units, and Set Up Map Projection

Files are stacked along the time dimension using `xr.open_mfdataset`. The raw NOHRSC 'Data' variable is in **meters**; here it is converted to **inches** for display. Latitude and longitude arrays are extracted as NumPy arrays for use with Matplotlib's `contourf`, which expects plain arrays rather than xarray coordinates.

A **Lambert Conformal Conic (LCC)** projection centered over the CONUS minimizes areal distortion for mid-latitude maps and matches the projection used in the SHELDUS plots for easy comparisons. Bounds are defined in geographic coordinates (`PlateCarree`) and transformed to LCC at the plot time.

In [ ]:
# stack all daily files in time dimension
data = xr.open_mfdataset(
    files,
    combine="nested", # combine files in nested structure
    concat_dim="time" # stack files along time dimension
)

# converting from meters to inches (1m = 39.3701in)
snow = data.Data * 39.3701

# extract coordinate arrays for matplotlib (contourf expects NumPy, not xarray)
lat = snow.lat.values
lon = snow.lon.values

# create lambert conformal projection
proj_lcc = ccrs.LambertConformal(
    central_longitude=-99,
    central_latitude=35,
    standard_parallels=[35]) # parallels for less distortion

# for conus or midwest extent
if map_extent == 'midwest':
    bound_n = 50.0
    bound_s = 35.0
    bound_w = -105.0
    bound_e = -80.0
elif map_extent == 'conus':
    bound_n = 50.0
    bound_s = 21.5
    bound_w = -122.0
    bound_e = -72.5

## 5. Compute Statistics & Plot

Each `plot_type` applies a different reduction along the `time` axis before plotting. A yellow-blue map is used for `std` and `days` since the absolute thresholds matter more than the relative spatial pattern. Here is a run down on the plot types:

| Plot Type | Reduction | Max Marker | Color Scale |
|-----------|-----------|------------|-------------|
| `sum`  | `np.sum` along time | yes | NWS diverging |
| `max`  | `np.max` along time | yes | NWS diverging |
| `std`  | `np.nanstd` along time (trace < 0.1 in masked) | no | YlGnBu |
| `days` | threshold leads to `np.nansum` | yes | YlGnBu |

- The `std` plot masks trace accumulations (< 0.1 in) as NaN before computing, otherwise near-zero values on snow-free days inflate the standard deviation.

In [ ]:
# shared color scale for sum/max
if plot_type == 'sum': # sum of snowfall
    snow_plot = snow.sum("time").values
    clevs = [0.5,1,2,3,4,6,8,12,16,20,24,30,36,48,60,72] # color levels optimized for major snow events
    cmap = col.ListedColormap(['#bdd7e7','#6baed6','#3182bd','#08519c','#082694','#ffff96','#ffc400','#ff8700','#db1400','#9e0000','#690000','#ccccff','#9f8cd8','#7c52a5','#561c72','#40dfff'])
    norm = col.BoundaryNorm(clevs,cmap.N)
    plot_title = "Cumulative Snow Accumulation (in)\n"
    # find location of maximum accumulation for marker placement
    max_val = np.nanmax(snow_plot)
    idx = np.where(snow_plot==max_val)
    iy, ix = idx[0][0], idx[1][0] # array indices
    max_lat = lat[iy]             # convert to lat/lon coordinates
    max_lon = lon[ix]

elif plot_type == 'max': # maximum 24-hour snowfall
    snow_plot = snow.max("time").values
    clevs = [0.5,1,2,3,4,6,8,12,16,20,24,30,36,48,60,72]
    cmap = col.ListedColormap(['#bdd7e7','#6baed6','#3182bd','#08519c','#082694','#ffff96','#ffc400','#ff8700','#db1400','#9e0000','#690000','#ccccff','#9f8cd8','#7c52a5','#561c72','#40dfff'])
    norm = col.BoundaryNorm(clevs,cmap.N)
    plot_title = "Maximum 24-Hour Snow Accumulation (in)\n"
    # find location of maximum accumulation for marker placement
    max_val = np.nanmax(snow_plot)
    idx = np.where(snow_plot==max_val)
    iy, ix = idx[0][0], idx[1][0]
    max_lat = lat[iy]
    max_lon = lon[ix]

elif plot_type == 'std': # standard deviation of snowfall, high for inconsistent snowfall, low for consistent snowfall over time period
    snow_temp = snow.values.copy()           # create a copy to avoid modifying the original
    snow_temp[snow_temp < 0.1] = np.nan      # mask trace amounts
    snow_plot = np.nanstd(snow_temp, axis=0) # calculate std along time axis
    clevs = np.arange(1, np.nanmax(snow_plot), 1)
    cmap = plt.cm.YlGnBu
    norm = col.BoundaryNorm(clevs, cmap.N)
    plot_title = "Standard Deviation of 24-Hour Snow Accumulation (in)\n"
    # note: no maximum marker for std plot (no single "max" location)

elif plot_type == 'days': # days of snow above specified threshold
    snow_plot = snow.values.copy()
    snow_plot[snow_plot < snow_thres] = 0.0  # below threshold = 0
    snow_plot[snow_plot >= snow_thres] = 1.0 # at/above threshold = 1
    snow_plot = np.nansum(snow_plot, axis=0) # sum = count of days
    clevs = np.arange(1, np.nanmax(snow_plot)+1, 1)
    cmap = plt.cm.YlGnBu
    norm = col.BoundaryNorm(clevs, cmap.N)
    plot_title = f"Days of 24-Hour Snowfall Above {snow_thres} inches\n"
    # find location with most days above threshold
    max_val = np.nanmax(snow_plot)
    idx = np.where(snow_plot==max_val)
    iy, ix = idx[0][0], idx[1][0]
    max_lat = lat[iy]
    max_lon = lon[ix]

# PLOT
fig, ax = plt.subplots(figsize=(12,8), dpi=125, subplot_kw={'projection': proj_lcc})
ax.set_extent([bound_w, bound_e, bound_s, bound_n], crs=ccrs.PlateCarree())

# background lakes and oceans
land_mask = cfeat.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor=cfeat.COLORS['land']) # land
sea_mask = cfeat.NaturalEarthFeature('physical', 'ocean', '50m', edgecolor='face', facecolor=cfeat.COLORS['water']) # ocean
lake_mask = cfeat.NaturalEarthFeature('physical', 'lakes', '50m', edgecolor='face', facecolor=cfeat.COLORS['water']) # lakes
ax.add_feature(sea_mask, zorder=0) 
ax.add_feature(land_mask, zorder=0)
ax.add_feature(lake_mask, zorder=0)

# plot snow data as filled contours
cs = ax.contourf(lon, lat, snow_plot, clevs, cmap=cmap, norm=norm, 
                 transform=ccrs.PlateCarree(), extend='max', zorder = 2) # transform=PlateCarree tells Cartopy the data is in lat/lon coordinates

# special handling for 'days' plot type that adds a contour line to show the boundary of where ANY threshold exceedance occurred
if plot_type == 'days':
    ax.contour(lon, lat, snow_plot, [1], colors='k', linewidths=0.2, 
               transform=ccrs.PlateCarree(), zorder=3)

# add political boundaries and geography on top of data
ax.coastlines(resolution='50m', color='black', linewidths=1.2, zorder = 3) 
state_borders = cfeat.NaturalEarthFeature(                                 
    category='cultural', name='admin_1_states_provinces_lakes',
    scale='50m', facecolor='none')
ax.add_feature(state_borders, edgecolor='black', linewidth=0.5, zorder = 4)
ax.add_feature(cfeat.BORDERS, linewidths=1.2, edgecolor='black', zorder =3)

# maximum snow marker (only for sum, max, and days)
if plot_type in ['sum', 'max', 'days']:
    if plot_type == 'days':
        label_text = 'Maximum Days:%s'%(str(int(max_val)))
    else:
        label_text = 'Maximum Snow:\n%0.1f"'%(max_val)
    ax.plot(max_lon, max_lat, '*', ms=22, mec='k', mew=2.7, fillstyle='none', 
            transform=ccrs.PlateCarree(), label=label_text, zorder=5)
# add legend for maximum marker
plt.legend(loc=3, prop={'size': 14})

# title
plot_title += dt.datetime.strftime(sdate,'%Y-%m-%d %H00 UTC') + " - " + dt.datetime.strftime(edate,'%Y-%m-%d %H00 UTC')
plt.title(plot_title, fontsize=16, fontweight='bold')

# add colorbar (matching the SHELDUS plot layout)
cax = fig.add_axes([0.12, 0.05, 0.76, 0.025])  # [left, bottom, width, height]
cbar = fig.colorbar(cs, cax=cax, orientation='horizontal', ticks=clevs)
cbar.set_label("Snow Accumulation (inches)", fontsize=12)

ax.set_xticks([])
ax.set_yticks([])

# save
fname = "SnowAccum" + dt.datetime.strftime(sdate,'%Y%m%d') + "_" + dt.datetime.strftime(edate,'%Y%m%d') + "_" + plot_type" + ".png"
save_path = Path("/home1/alamia/Research/SnowAccum/") / fname
plt.savefig(save_path, dpi=100, bbox_inches='tight')
print(f"Saved to: {os.path.abspath(save_path)}")

plt.show()
plt.close()
data.close()